In [1]:
import numpy as np

def Zcen2quad(zx, zy, zz, sfrq, ival):
    # Declare variables
    sort = np.zeros(3, dtype=int)
    qpas = np.zeros(3)
    qgon13 = np.zeros(2)
    qgon23 = np.zeros(2)
    qgon11 = np.zeros(2)
    qgon22 = np.zeros(2)
    qgon33 = np.zeros(2)
    a1m3 = np.zeros(2)
    test = np.zeros(2)
    qgon = np.zeros((3, 3))
    temp = np.zeros((3, 3))

    # Early return condition (translated from the original condition)
    if np.abs(zx[4]) < -1e5 and np.abs(zy[4]) < -1e5 and np.abs(zz[4]) < -1e5 and \
       np.abs(zx[5]) < -1e5 and np.abs(zy[5]) < -1e5 and np.abs(zz[5]) < -1e5:
        print("Not in the mood for determining quad and CSA from the central transition")
        return None, None

    # Constants and calculations
    k = 16.0 * sfrq / (9.0 * (4.0 * ival * (ival + 1.0) - 3.0)) / 1.0E3  # [qgon] = MHz
    qgon[0, 1] = np.sqrt(k * (np.sqrt(zz[4] ** 2 + zz[5] ** 2) - zz[4]))
    a1m2 = np.sqrt(4.0 * k * (np.sqrt(zz[4] ** 2 + zz[5] ** 2) + zz[4]))
    qgon13[0] = np.sqrt(k * (np.sqrt(zy[4] ** 2 + zy[5] ** 2) - zy[4]))
    qgon13[1] = -np.sqrt(k * (np.sqrt(zy[4] ** 2 + zy[5] ** 2) - zy[4]))
    a2m3 = np.sqrt(4.0 * k * (np.sqrt(zx[4] ** 2 + zx[5] ** 2) + zx[4]))

    for i in range(2):
        qgon23[i] = np.sqrt(k * (np.sqrt(zx[4] ** 2 + zx[5] ** 2) - zx[4]))
        a1m3[i] = np.sqrt(4.0 * k * (np.sqrt(zy[4] ** 2 + zy[5] ** 2) + zy[4]))

    # Sign determination
    a1m2 *= 1.0 if zz[5] >= 0.0 else -1.0

    # Adjust signs and test
    for i in range(2):
        a1m3[i] *= 1.0 if zy[5] * qgon13[i] >= 0.0 else -1.0
        test[i] = zx[5] / (zy[5] / qgon13[i] - zz[5] / qgon[0, 1])
        qgon23[i] *= -1.0 if test[i] < 0.0 else 1.0

        qgon11[i] = (a1m2 + a1m3[i]) / 3.0
        qgon22[i] = (a1m3[i] - 2.0 * a1m2) / 3.0
        qgon33[i] = (a1m2 - 2.0 * a1m3[i]) / 3.0
        test[i] = np.abs(np.abs(qgon22[i] - qgon33[i]) / a2m3 - 1.0)

    # Choose the best option
    i = 1 if test[1] < test[0] else 0
    qgon[0, 0] = qgon11[i]
    qgon[0, 2] = qgon13[i]
    qgon[1, 1] = qgon22[i]
    qgon[1, 2] = qgon23[i]
    qgon[2, 2] = qgon33[i]

    # Copy qgon to temp and diagonalize
    temp = np.copy(qgon)
    eigvals, rq = np.linalg.eigh(temp)
    qpas = eigvals
    sort = np.argsort(qpas)

    if qpas[2] < 0.0:
        qgon = -qgon
        temp = -temp
        eigvals, rq = np.linalg.eigh(temp)
        qpas = eigvals
        sort = np.argsort(qpas)

    # Sort and optimize rq
    rq = rq[:, sort]
    if np.linalg.det(rq) < 0:
        rq[:, 2] = -rq[:, 2]

    # Calculate cq and etaq
    cq = 2.0 * ival * (2.0 * ival - 1.0) * qpas[2]
    etaq = (qpas[1] - qpas[0]) / qpas[2]

    return cq, etaq
